In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser, ListOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel
from typing import List

model = ChatOllama(model="gpt-oss:120b-cloud")

In [ ]:
prompt = PromptTemplate.from_template("When India got independence?")

response = model.invoke(prompt.format())

print(type(response))
print(response)

In [ ]:
string_parser = StrOutputParser()

prompt = PromptTemplate(
    template="When {country} got independence?",
    input_variables=["country"]
)


chain = prompt | model | string_parser
response = chain.invoke({"country": "India"})
print(type(response))
print(response)

In [ ]:
class CityListOutputParser(ListOutputParser):
    def parse(self, text: str) -> list:
        # Split the text by , and remove whitespace
        cities = [city.strip() for city in text.split(",") if city.strip()]
        return cities
prompt = PromptTemplate(
    template="Give me a list comma separated only names of top 5 cities in the world by their population"
)
list_parser = CityListOutputParser()
chain = prompt | model | list_parser
response = chain.invoke({})
print(type(response))
print(response)

In [ ]:
class GlobalWarmingSeverityParser(BaseModel):
    description: str
    hashtag: str

json_parser = JsonOutputParser(pydantic_object=GlobalWarmingSeverityParser)
prompt = PromptTemplate(
    template="#Format: {format_instruction}\nQuery:Explain the severity of global warming",
    partial_variables={"format_instruction": json_parser.get_format_instructions()}
)

chain = prompt | model | json_parser
response = chain.invoke({})
print(type(response))
print(response)

In [ ]:
class Person(BaseModel):
    name: str
    age: int
    hobbies: List[str]
person_parser = PydanticOutputParser(pydantic_object=Person)
prompt = PromptTemplate(
    template="#Format: {format_instruction}\nQuery: Tom is 20 years old and he likes playing cricket, football and swimming",
    partial_variables={"format_instruction": person_parser.get_format_instructions()}
)

chain = prompt | model | person_parser
response = chain.invoke({})
print(type(response))
print(response)